# Single guiding-centre orbit with BM4

This experiment evolves one guiding centre in a reproducible periodic potential. The BM4 direct/adjoint composition uses the coupled extended GC formulation, and the animation shows the computed orbit over the gyroaveraged potential.

All potential, physical, initial-condition, formulation, and numerical parameters are explicit below.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation. BM4Midpoint applies arithmetic averaging after the complete composition.


In [ ]:
%matplotlib inline

import numpy as np

from dynamics import GuidingCenterDynamics
from initial_conditions import GCInitialConfiguration
from simulation import (
    BM4Midpoint,
    InitialValueProblem,
    SimulationRequest,
    simulate,
)
from studies import RandomPotentialConfig
from visualization import (
    animate_gc_particle_solution,
    display_animation,
)

## Reproducible configuration

`rho` sets the normalized Larmor radius used to gyroaverage the potential. `coupling_frequency` is the harmonic mixing frequency of the doubled-state GC formulation used by BM4.

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=16,
    nx=48,
    ny=48,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

rho = 0.3
initial_position = (np.pi, np.pi)
coupling_frequency = np.pi / 8

t_span = (0.0, 4 * np.pi)
max_step = 0.01
output_sample_count = 241

trajectory = GCInitialConfiguration.from_components(
    x=np.asarray([initial_position[0]]),
    y=np.asarray([initial_position[1]]),
    
)
dynamics = GuidingCenterDynamics(potential, rho=rho)
problem = InitialValueProblem(dynamics, trajectory)
method = BM4Midpoint(coupling_frequency=coupling_frequency)
request = SimulationRequest.uniform(
    t_span=t_span,
    max_step=max_step,
    sample_count=output_sample_count,
)

print(potential_config)
print(f"GC coupling frequency: {coupling_frequency:.6f}")

## BM4 integration

In [ ]:
solution = simulate(problem, method, request)

print(f"Fixed BM4 steps: {solution.diagnostics['step_count']}")

## Animated guiding-centre orbit

The background is the effective potential used by the GC dynamics, rather than the unaveraged input potential. The black curve is the orbit accumulated up to the current frame.

In [ ]:
animation = animate_gc_particle_solution(
    dynamics.effective_potential,
    solution,
    frames=61,
    interval=80,
    cmap="RdBu_r",
    repeat=True,
)

display_animation(animation, embed_limit_mb=30.0)